# 第1回：予測モデルを動かしてみる

**今日の問い：予測モデルは、データを受け取って何を返しているのか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 特徴量・目的変数・学習・予測を、画面上の入出力と結びつける
- 予測を関数へ切り出し、型ヒントとassertで最小の検証を付ける
- ベースラインと比べ、設定変更の効果を交差検証の平均とばらつきで語る

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- 特徴量：予測時点でモデルへ渡す情報
- 目的変数：予測したい答え
- 学習：既知データから関係を推定する処理
- 推論：学習済みモデルを未知データへ使う処理
- 並べ替え重要度：列を崩したときの性能低下で測る寄与

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


## 予測モデルを、料理ではなく実験でたとえる

「機械学習モデル」と聞くと難しく感じますが、やっていることは研究者の頭の中と似ています。
過去にたくさんの実験（データ）を見て「この条件なら活性が出やすい」という**経験則**を作り、
未知の条件に対して「たぶん活性あり／なし」を答える。この経験則づくりが**学習（fit）**、
未知への当てはめが**予測（predict）**です。

この回では、中身のアルゴリズムは一旦置いて、**何を入れると何が返るか**だけを体で覚えます。
下に出てくる言葉を、実験のイメージと結びつけておきましょう。

| 言葉 | 意味 | 実験でのイメージ |
|---|---|---|
| 特徴量（X） | モデルへ渡す入力の列 | 分子量・LogPなど、計画時に分かっている条件 |
| 目的変数（y） | 予測したい答えの列 | その条件で活性が出たか（0/1） |
| 学習（fit） | 過去データから関係を推定する | 過去の実験ノートを読み込む |
| 予測（predict） | 学習済みモデルを未知へ使う | 新しい条件の結果を見立てる |


## まずデータを開く

分析は「データを見る」ことから始まります。次のセルはCSV（表計算のような表データ）を読み込み、
`df`という名前の**表（DataFrame）**に入れます。`df.head()`は先頭5行だけを表示します。
全部で何行・何列あるかも一緒に出します。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


### 出力の読み方

- `420行 × 19列`：試料が420件、各試料について19種類の情報がある、という意味です。
- 表の**1行が1試料**、**1列が1種類の情報**です。`sample_id`は試料の名札で、予測には使いません。
- `NaN`（Not a Number）は**欠損＝その値が測られていない**印です。第3〜4回で詳しく扱います。

まだ意味が分からない列があっても大丈夫です。今日は下の5列だけ使います。


## モデルへ渡す列を決めて、学習させる

ここが今日の中心です。次のセルは4つの手順を続けて行っています。1行ずつ何をしているかは、
セルの下の「コードの読み方」で説明します。まず実行して、出てくる数字を眺めてください。

**なぜ「ベースライン」と比べるのか？** いきなり高機能なモデルの点数だけ見ても、それが
「すごい」のか「当たり前」なのか分かりません。そこで、**いつも多数派（ここでは非活性）と
答えるだけの単純なモデル**を先に用意し、本命がそれをどれだけ上回るかで価値を測ります。
これは「対照実験（コントロール）」と同じ考え方です。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

features = ["molecular_weight", "logp", "tpsa", "h_bond_donors", "rotatable_bonds"]
X = df[features].fillna(df[features].median())
y = df["active"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
model = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=42).fit(X_train, y_train)
for name, estimator in {"多数派ベースライン": baseline, "Random Forest": model}.items():
    pred = estimator.predict(X_valid)
    print(f"{name:14s} accuracy={accuracy_score(y_valid, pred):.3f}  F1={f1_score(y_valid, pred):.3f}")


### コードの読み方（1行ずつ）

- `features = [...]`：モデルへ渡す**入力列の名前リスト**。ここでは分子の性質5つを選びました。
- `X = df[features].fillna(...)`：`X`は入力の表。`fillna(...median())`は、欠損を**その列の中央値で埋める**処理です（モデルは空欄を扱えないため）。
- `y = df["active"]`：`y`は答えの列（活性=1／非活性=0）。
- `train_test_split(...)`：データを**学習用（train）と検証用（valid）に分ける**関数。`test_size=0.25`で25%を検証用に取り置きます。未知データでの成績を測るため、検証用は学習に使いません。`random_state=42`は分け方を固定して**毎回同じ結果**にするおまじない、`stratify=y`は活性の割合が両側で揃うようにする指定です。
- `.fit(X_train, y_train)`：**学習**。過去データ（train）から関係を覚えます。
- `.predict(X_valid)`：覚えた関係を**検証用の未知データ**へ当てはめて予測します。

### 出力の読み方

- **accuracy（正解率）**：全体のうち何割を当てたか。
- **F1**：活性を「見つける力」と「間違えない力」のバランス（0〜1、高いほど良い）。活性が少ないデータでは正解率より頼りになります（第8回で詳説）。
- 見るべきは**Random Forestがベースラインをどれだけ上回ったか**。差が小さいなら、そのモデルはまだ価値を出せていません。


## TRY：たった1試料を予測させてみる

モデルは表全体だけでなく、**1件ずつ**予測できます。検証用データの先頭1件を渡してみましょう。
`predict`は0か1の**判定**を、`predict_proba`は**活性である確率**を返します。


In [ ]:
one_sample = X_valid.iloc[[0]]
display(one_sample)
print("予測クラス:", model.predict(one_sample)[0])
print("活性である確率:", round(model.predict_proba(one_sample)[0, 1], 3))


### 出力の読み方と、よくある勘違い

- 上の表がこの試料の**入力（特徴量）**、その下がモデルの**答え**です。
- **予測クラス**が`1`なら「活性ありと判定」、`0`なら「非活性と判定」。
- **確率0.8**は「80%の確信で活性」という**モデルの自信**であって、「必ず活性」という保証ではありません。ここを混同しないことが、今日いちばん大事な感覚です。
- `iloc[[0]]`と二重角括弧にしているのは、1行でも**表の形のまま**渡すためです（`iloc[0]`だと1次元になり、モデルが受け取れません）。


## CORE深掘り：同じ評価は「関数」にまとめる

上では `accuracy_score(...)` と `f1_score(...)` を手で並べました。同じ評価を何度も書くと、
書き間違いが起きます。そこで**名前を付けた処理のかたまり（関数）**にまとめます。

- `def evaluate_classifier(...) -> dict:` の `-> dict` は「この関数は辞書を返す」という**型ヒント**（読み手への注釈）。
- 関数の1行目の文字列は**docstring**で、何をする関数かの説明です。
- `assert 条件, "メッセージ"` は「この条件が成り立たなければ止まれ」という**自己点検**。想定外の値が返っていないかを自動で見張ります。


In [ ]:
def evaluate_classifier(estimator, X_valid, y_valid) -> dict:
    "検証データでaccuracyとF1を計算し、辞書で返す純粋な評価関数。"
    pred = estimator.predict(X_valid)
    return {
        "accuracy": round(accuracy_score(y_valid, pred), 3),
        "f1": round(f1_score(y_valid, pred), 3),
    }

scores = evaluate_classifier(model, X_valid, y_valid)
assert set(scores) == {"accuracy", "f1"}, "返す指標が想定と違います"
assert 0.0 <= scores["f1"] <= 1.0, "F1は0〜1のはず"
scores


### なぜ関数にすると良いのか

- **繰り返しに強い**：別のモデルを評価したいとき、`evaluate_classifier(別のモデル, ...)`と呼ぶだけ。
- **間違いに気づける**：`assert`があるので、うっかりF1が1.2のような有り得ない値になったら即座に止まります。
- **読みやすい**：中身を知らなくても関数名で「何をするか」が伝わります。

この「小さく作って、テストで守る」考え方は第2回でさらに練習します。


## CHANGE：1か所だけ変えて、違いを観察する

`max_depth=4`（木の深さ）を`2`や`8`に変えて、上のセルを再実行してみましょう。
深くすると学習データには合いますが、検証スコアは必ずしも上がりません（**過学習**）。
変えた値・理由・結果を1行でメモしておきます。

## ASK COPILOT

M365 Copilotに、`fit`と`predict_proba`の違いを「測定装置の校正」と「未知試料の測定」に
たとえて説明してもらいましょう。返答を鵜呑みにせず、上の出力と照らして確かめます。

## まとめ

- 表の1行＝1試料、列＝情報。**特徴量（X）**を入れ、**目的変数（y）**を予測する。
- **fit=学習、predict=予測**。確率は「自信」であって真実ではない。
- 良し悪しは**ベースラインとの差**で測り、評価は**関数**にまとめて再利用する。


## DEEP DIVE：木の深さと「過学習」を交差検証で見る

ここからは経験者・自習向けの発展です。1回の学習/検証の分け方だと、たまたま簡単な検証データに
当たって点数が良く見えることがあります。そこで**交差検証**を使います。

**交差検証（cross validation）とは**：データを5つに分け、「4つで学習→残り1つで検証」を
担当を変えて5回行い、5回のスコアを平均する方法です。1回だけの運・不運をならして、
より信頼できる成績を出します。

次の表では、木の深さ（`max_depth`）を変えながら、**学習F1**と**検証F1**の両方を出します。
学習F1だけが高くて検証F1が伸びない＝**過学習**（覚えすぎて未知に弱い）のサインです。


In [ ]:
import pandas as pd
from sklearn.model_selection import cross_validate, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for depth in [1, 2, 3, 4, 6, 8, None]:
    estimator = RandomForestClassifier(n_estimators=200, max_depth=depth, random_state=42)
    result = cross_validate(estimator, X, y, cv=cv, scoring="f1", return_train_score=True)
    rows.append({
        "max_depth": str(depth),
        "学習F1": result["train_score"].mean(),
        "検証F1": result["test_score"].mean(),
        "検証F1_SD": result["test_score"].std(),
    })
pd.DataFrame(rows).round(3)


### 出力の読み方

- 上から下へ木を深くすると、**学習F1はほぼ単調に上がる**はずです（覚える力が増えるため）。
- 一方**検証F1**はどこかで頭打ち・悪化します。その手前が「ちょうど良い深さ」の目安です。
- `検証F1_SD`は5回のばらつき。小さいほど安定。**平均が少し高くてもSDが大きいモデル**は、運任せに近いので注意します。


### 特徴量重要度は2種類を見比べる

「どの特徴量が効いているか」を知りたくなります。ただし木モデルが標準で出す**不純度重要度**は、
値の種類が多い列を過大評価する癖があります。そこで、**列の値をわざと混ぜて性能がどれだけ落ちるか**で
測る**並べ替え重要度（permutation importance）**と並べて読みます。落ち幅が大きい列ほど本当に効いています。


In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(model, X_valid, y_valid, scoring="f1", n_repeats=20, random_state=42)
importance = pd.DataFrame({
    "特徴量": features,
    "不純度重要度": model.feature_importances_,
    "並べ替え重要度": perm.importances_mean,
    "並べ替えSD": perm.importances_std,
}).sort_values("並べ替え重要度", ascending=False)
importance.round(3)


### 出力の読み方

2つの列で順位が食い違ったら、**並べ替え重要度**を優先します。並べ替え重要度が0付近（SDより小さい）なら、
その特徴量は「効いているとは言い切れない」と読みます。


## CHALLENGE：確率は「当たっている」か（較正）

モデルが「確率0.8」と言った試料たちは、本当に約80%が活性でしょうか。確率を確率帯ごとに束ね、
**その帯の実際の活性率**と見比べます。予測確率と実際がだいたい一致していれば、確率を意思決定に
使えます（この一致度を**較正**と呼び、第8回で詳しく扱います）。


In [ ]:
probability = model.predict_proba(X_valid)[:, 1]
bucket = pd.cut(probability, bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0])
calibration = (
    pd.DataFrame({"確率帯": bucket, "実際の活性": y_valid.to_numpy()})
    .groupby("確率帯", observed=True)["実際の活性"]
    .agg(件数="size", 実際の活性率="mean")
)
calibration.round(3)


### 出力の読み方

各行は「その確率帯に入った試料の件数」と「実際に活性だった割合」です。
`0.6〜0.8`の帯で実際の活性率が0.7前後なら、確率はよく較正されています。大きくずれていたら、
確率の数字を鵜呑みにせず、順位付け（どれを先に試すか）にとどめる使い方が安全です。
なお件数が少ない帯は割合が不安定なので、件数も一緒に見ます。


## よくある誤り

- 学習データの成績を実力だと思う
- 1試料の予測だけでモデル全体を判断する
- 良い数値が出るまで設定を無計画に変える

## SELF-STUDY（任意・30〜60分）

- evaluate_classifierを拡張し、precisionとrecallも返してテストを足す
- 木の深さ2・4・8を交差検証で比較し、選ぶ理由を平均とばらつきで2文書く

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. Xとyはそれぞれ何か
2. 不純度重要度と並べ替え重要度はどう違うか
3. 単一の検証スコアより交差検証を見る理由は何か

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
